# Loading the Data in

In [1]:
import numpy as np
import nibabel as nib
from scipy.ndimage import binary_fill_holes, binary_dilation, binary_erosion
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

# Data paths
bmode_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_Fund_RAW.nii'
ceus_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/combined_nifti/p16_wk12_CHI_RAW.nii'
small_mc_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/p16_wk12_small_mc_roi.nii.gz'
large_roi_path = '/Users/samantha/Desktop/ultrasound lab stuff/raw_ctdna/p16/wk12/p16_wk12_static_roi_139.nii.gz'

# Load data
bmode_data = nib.load(bmode_path).get_fdata()
ceus_data = nib.load(ceus_path).get_fdata()
small_mc_roi_seg = nib.load(small_mc_roi_path).get_fdata()
large_roi_seg = nib.load(large_roi_path).get_fdata()

# Process ROIs for CEUS Analysis

In [2]:
target_shape = bmode_data.shape

# Process small MC ROI
if small_mc_roi_seg.shape[0] == 1048:
    mid = 524
    left_half = small_mc_roi_seg[:mid, :, :]
    right_half = small_mc_roi_seg[mid:, :, :]
    small_mc_roi_seg = left_half if np.sum(left_half > 0) > np.sum(right_half > 0) else right_half
    if small_mc_roi_seg.shape != target_shape:
        small_mc_roi_seg = small_mc_roi_seg.transpose(1, 0, 2)

# Process large ROI
if large_roi_seg.shape[0] == 1048:
    mid = 524
    left_half = large_roi_seg[:mid, :, :]
    right_half = large_roi_seg[mid:, :, :]
    large_roi_seg = left_half if np.sum(left_half > 0) > np.sum(right_half > 0) else right_half
    if large_roi_seg.shape != target_shape:
        large_roi_seg = large_roi_seg.transpose(1, 0, 2)

n_frames = bmode_data.shape[2]

# Motion Tracking and Compensation

In [3]:
reference_frame = 139

# Track top-left corner of small MC ROI
top_left_positions = np.zeros((n_frames, 2))
translation_vectors = np.zeros((n_frames, 3))
frames_with_roi = []

for t in range(n_frames):
    mask_t = small_mc_roi_seg[:, :, t] > 0
    if np.sum(mask_t) == 0:
        top_left_positions[t] = [-1, -1]
        continue
    frames_with_roi.append(t)
    y_indices, x_indices = np.where(mask_t)
    top_left_positions[t] = [y_indices.min(), x_indices.min()]

In [4]:
# Interpolate missing frames
for t in range(n_frames):
    if top_left_positions[t, 0] == -1:
        valid_before = [f for f in frames_with_roi if f < t]
        valid_after = [f for f in frames_with_roi if f > t]
        if valid_before and valid_after:
            t_before, t_after = max(valid_before), min(valid_after)
            weight = (t - t_before) / (t_after - t_before)
            top_left_positions[t] = (1 - weight) * top_left_positions[t_before] + weight * top_left_positions[t_after]
        elif valid_before:
            top_left_positions[t] = top_left_positions[max(valid_before)]
        elif valid_after:
            top_left_positions[t] = top_left_positions[min(valid_after)]

In [5]:
# Calculate translation vectors relative to reference frame
ref_top_left = top_left_positions[reference_frame]
for t in range(n_frames):
    dy = top_left_positions[t, 0] - ref_top_left[0]
    dx = top_left_positions[t, 1] - ref_top_left[1]
    translation_vectors[t] = [dx, dy, t]

In [6]:
# Fill large ROI if needed (check if it's just a border)
test_frame = min(250, n_frames - 1)
roi_test = large_roi_seg[:, :, test_frame]
if np.sum(roi_test > 0) > 0:
    y_coords, x_coords = np.where(roi_test > 0)
    bbox_area = (y_coords.max() - y_coords.min() + 1) * (x_coords.max() - x_coords.min() + 1)
    fill_percentage = (np.sum(roi_test > 0) / bbox_area) * 100
    
    if fill_percentage < 30:
        large_roi_seg_filled = np.zeros_like(large_roi_seg)
        for t in range(n_frames):
            roi_frame = large_roi_seg[:, :, t]
            if np.sum(roi_frame > 0) > 0:
                dilated = binary_dilation(roi_frame > 0, iterations=2)
                filled = binary_fill_holes(dilated)
                filled = binary_erosion(filled, iterations=2)
                large_roi_seg_filled[:, :, t] = filled.astype(np.uint8)
        large_roi_seg = large_roi_seg_filled

In [7]:
# Apply motion compensation to large ROI
large_roi_mc = np.zeros_like(large_roi_seg)
for t in range(n_frames):
    shift_y = int(round(translation_vectors[t, 1]))
    shift_x = int(round(translation_vectors[t, 0]))
    shifted = large_roi_seg[:, :, t].copy()
    
    if shift_y != 0:
        shifted = np.roll(shifted, shift_y, axis=0)
        if shift_y > 0:
            shifted[:shift_y, :] = 0
        else:
            shifted[shift_y:, :] = 0
    if shift_x != 0:
        shifted = np.roll(shifted, shift_x, axis=1)
        if shift_x > 0:
            shifted[:, :shift_x] = 0
        else:
            shifted[:, shift_x:] = 0
    
    large_roi_mc[:, :, t] = shifted

# Apply Motion Compensation to Image Data

In [8]:
import os

# Use float arrays so out-of-bounds pixels can be NaN
bmode_data_mc = np.full_like(bmode_data, np.nan, dtype=float)
ceus_data_mc = np.full_like(ceus_data, np.nan, dtype=float)

for t in range(n_frames):
    shift_y = -int(round(translation_vectors[t, 1]))
    shift_x = -int(round(translation_vectors[t, 0]))

    shifted_bmode = bmode_data[:, :, t].astype(float)
    shifted_ceus = ceus_data[:, :, t].astype(float)

    if shift_y != 0:
        shifted_bmode = np.roll(shifted_bmode, shift_y, axis=0)
        shifted_ceus = np.roll(shifted_ceus, shift_y, axis=0)
    if shift_x != 0:
        shifted_bmode = np.roll(shifted_bmode, shift_x, axis=1)
        shifted_ceus = np.roll(shifted_ceus, shift_x, axis=1)

    # NaN out the wrapped (out-of-bounds) border pixels
    if shift_y > 0:
        shifted_bmode[:shift_y, :] = np.nan
        shifted_ceus[:shift_y, :] = np.nan
    elif shift_y < 0:
        shifted_bmode[shift_y:, :] = np.nan
        shifted_ceus[shift_y:, :] = np.nan
    if shift_x > 0:
        shifted_bmode[:, :shift_x] = np.nan
        shifted_ceus[:, :shift_x] = np.nan
    elif shift_x < 0:
        shifted_bmode[:, shift_x:] = np.nan
        shifted_ceus[:, shift_x:] = np.nan

    bmode_data_mc[:, :, t] = shifted_bmode
    ceus_data_mc[:, :, t] = shifted_ceus

# Crop to Valid (In-Bounds) Region

In [9]:
# Find the tightest crop box that is valid (non-NaN) across ALL frames
# A pixel position is valid in a frame if the MC shift didn't push it out of bounds.
# We take the intersection across all frames so the cropped volume is fully defined everywhere.

valid_mask = np.all(np.isfinite(ceus_data_mc), axis=2)  # (H, W) — True where always valid

valid_rows = np.where(valid_mask.any(axis=1))[0]
valid_cols = np.where(valid_mask.any(axis=0))[0]

row_min, row_max = valid_rows[0], valid_rows[-1] + 1
col_min, col_max = valid_cols[0], valid_cols[-1] + 1

print(f"Valid crop region: rows [{row_min}:{row_max}], cols [{col_min}:{col_max}]")
print(f"Original shape: {ceus_data_mc.shape[:2]}, Cropped shape: ({row_max-row_min}, {col_max-col_min})")

# Crop image data and ROI to the valid region
bmode_data_mc_crop = bmode_data_mc[row_min:row_max, col_min:col_max, :]
ceus_data_mc_crop = ceus_data_mc[row_min:row_max, col_min:col_max, :]
large_roi_mc_crop = large_roi_mc[row_min:row_max, col_min:col_max, :]

# Store crop bounds for reference (e.g. mapping back to original coordinates)
crop_bounds = (row_min, row_max, col_min, col_max)

Valid crop region: rows [28:589], cols [119:465]
Original shape: (604, 524), Cropped shape: (561, 346)


# Save CEUS MC 

In [ ]:
original_ceus = nib.load(ceus_path)

# Save the cropped MC data (NaN-free, valid region only)
ceus_mc_nii = nib.Nifti1Image(ceus_data_mc_crop, affine=original_ceus.affine, header=original_ceus.header)
output_path = os.path.join(os.path.dirname(small_mc_roi_path), "image_mc_data.nii.gz")
nib.save(ceus_mc_nii, output_path)
print(f"Saved cropped CEUS MC to: {output_path}")

# Visualization and Video Export

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import ipywidgets as widgets
from IPython.display import display, clear_output

vmin_bmode, vmax_bmode = np.nanpercentile(bmode_data_mc, [1, 99])
vmin_ceus, vmax_ceus = np.nanpercentile(ceus_data, [1, 99])

fig_export, axes_export = plt.subplots(1, 3, figsize=(18, 6))
plt.close(fig_export)

roi_ref = large_roi_seg[:, :, reference_frame]

def draw_frame(frame_idx):
    for ax in axes_export:
        ax.clear()

    # Mask ROI contour to only show where image data is valid (non-NaN)
    bmode_valid = np.isfinite(bmode_data_mc[:, :, frame_idx])
    ceus_valid = np.isfinite(ceus_data_mc[:, :, frame_idx])
    roi_bmode = roi_ref * bmode_valid
    roi_ceus = roi_ref * ceus_valid

    axes_export[0].imshow(ceus_data[:, :, frame_idx], cmap='gray', vmin=vmin_ceus, vmax=vmax_ceus)
    axes_export[0].contour(roi_ref, colors='red', linewidths=1.5)
    axes_export[0].set_title('Original CEUS', fontsize=14)
    axes_export[0].axis('off')

    axes_export[1].imshow(bmode_data_mc[:, :, frame_idx], cmap='gray', vmin=vmin_bmode, vmax=vmax_bmode)
    if roi_bmode.any():
        axes_export[1].contour(roi_bmode, colors='red', linewidths=1.5)
    axes_export[1].set_title('B-mode (MC)', fontsize=14)
    axes_export[1].axis('off')

    axes_export[2].imshow(ceus_data_mc[:, :, frame_idx], cmap='gray', vmin=vmin_ceus, vmax=vmax_ceus)
    if roi_ceus.any():
        axes_export[2].contour(roi_ceus, colors='red', linewidths=1.5)
    axes_export[2].set_title('CEUS (MC)', fontsize=14)
    axes_export[2].axis('off')

    dx = translation_vectors[frame_idx, 0]
    dy = translation_vectors[frame_idx, 1]
    fig_export.suptitle(f'Frame {frame_idx}  |  dx={dx:.1f}, dy={dy:.1f} px', fontsize=16, fontweight='bold')
    fig_export.tight_layout()

def on_frame_change(change):
    with preview_output:
        clear_output(wait=True)
        draw_frame(change['new'])
        display(fig_export)

def do_export(btn):
    video_path = os.path.join(os.path.dirname(small_mc_roi_path), "image_correction.mp4")
    export_btn.disabled = True
    export_btn.description = 'Exporting...'
    with export_output:
        clear_output()
        writer = FFMpegWriter(fps=10)
        with writer.saving(fig_export, video_path, dpi=100):
            for t in range(n_frames):
                draw_frame(t)
                writer.grab_frame()
        print(f"Saved to: {video_path}")
    export_btn.disabled = False
    export_btn.description = 'Export Video'

frame_slider = widgets.IntSlider(value=reference_frame, min=0, max=n_frames-1, step=1, description='Frame:', continuous_update=False)
export_btn = widgets.Button(description='Export Video', button_style='success')
preview_output = widgets.Output()
export_output = widgets.Output()

frame_slider.observe(on_frame_change, names='value')
export_btn.on_click(do_export)

display(widgets.HBox([frame_slider, export_btn]), preview_output, export_output)
frame_slider.value = reference_frame

Output()

Output()